# ETHICS Religion Persona — Analisis Akurasi v4 (Persona-First)

Notebook ini adalah lapisan penyajian di atas bundle v3 agama yang tervalidasi.
Inferensi tidak dihitung ulang. Akurasi penuh dan seluruh nilai uji statistik
dibaca dari tabel v3; status respons dan token dihitung sebagai ringkasan
deskriptif dari CSV sumber yang dirujuk manifest v3. Empat subset—Commonsense,
Deontology, Justice, dan Virtue—selalu dipertahankan terpisah.

## 0. Provenance dan validation gate

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

start = Path.cwd().resolve()
REPO_ROOT = next(path for path in [start, *start.parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.MAIN_PARSING.ETHICS.build_religion_publication_notebook_v4_accuracy_persona_first import run_analysis

result = run_analysis()
OUTPUT_DIR = Path(result["output_dir"])
FIGURE_DIR = OUTPUT_DIR / "figures"
display(pd.DataFrame([result["validation"]]).T.rename(columns={0: "value"}))
assert result["validation"]["valid"]
assert result["validation"]["api_calls"] == 0
assert result["validation"]["statistics_recomputed"] is False

## 1. Akurasi N, E, I1, I2, dan I3 untuk setiap subset

Tabel berikut memuat seluruh 408 kombinasi model × kondisi × subset. `accuracy_full`
berasal dari tabel v3 dengan seluruh respons sebagai penyebut. `accuracy_committed`
ditampilkan sebagai metrik tambahan dan tidak menggantikan akurasi penuh.

In [ ]:
pd.set_option("display.max_rows", 1000)
display(result["accuracy_all_conditions"])

## 2. Perbandingan antarpersona agama — hasil utama

In [ ]:
display(result["persona_summary_by_model"])
display(result["persona_summary"])
display(Image(filename=str(FIGURE_DIR / "04_religion_pairwise_e_effect_holm.png"), width=1500))

Urutan orientasi pasangan pada tabel dan grafik adalah Islam, Kristen, Yahudi,
lalu Ateis. Jika orientasi sumber v3 dibalik, builder hanya membalik delta,
Cohen's h, dan arah discordant pair; nilai-p tidak diubah.

## 3. Perbandingan I1–E, I2–E, dan I3–E

In [ ]:
display(result["scenario_summary"].query("comparison_category == 'implicit_vs_explicit'"))
display(Image(filename=str(FIGURE_DIR / "02_religion_implicit_vs_explicit_effect_q_holm.png"), width=1500))

## 4. Perbandingan E–N, I1–N, I2–N, dan I3–N

In [ ]:
display(result["scenario_summary"].query("comparison_category == 'persona_vs_neutral'"))
display(Image(filename=str(FIGURE_DIR / "03_religion_persona_vs_neutral_effect_q_holm.png"), width=1500))

## 5. Akurasi penuh, committed, commit rate, abstain, dan missing

Status dihitung per model × kondisi × subset. `abstain` dan `missing` tetap
berada di dalam denominator akurasi penuh dan bernilai tidak benar sesuai
manifest v3. Perbedaan tingkat komitmen tidak boleh dibaca sebagai perbedaan
akurasi tanpa melihat kolom status pada tabel yang sama.

In [ ]:
display(result["status"])
display(result["status_by_model"])
display(result["status_by_subset"])
display(Image(filename=str(FIGURE_DIR / "05_religion_response_status_by_model.png"), width=1200))

## 6. Input token, completion token, reasoning token, dan total token

Jumlah valid dan rata-rata token ditampilkan tanpa imputasi. `n_valid_*` dapat
lebih kecil daripada `total`; metadata token yang hilang tetap hilang.

In [ ]:
display(result["tokens"])
display(result["token_summary_by_model_condition"])
display(Image(filename=str(FIGURE_DIR / "06_religion_total_tokens_by_condition.png"), width=1500))

## 7. Batas interpretasi subset

`Commonsense` memakai label 1 untuk tindakan salah, sedangkan `Justice` memakai
label 1 untuk pernyataan wajar. Pada `Deontology`, angka yang tampil adalah
kecocokan label per butir karena informasi penentu *excuse* tidak dikirim kepada
model. Pada `Virtue`, angka juga hanya kecocokan label per butir karena bentuk
pertanyaan tidak sepenuhnya cocok dengan konstruk label. Karena itu tidak ada
satu angka akurasi gabungan empat subset sebagai hasil inferensial.

## 8. Indeks artefak dan kontrak reproduksi

In [ ]:
display(result["output_index"])